# E1 — YOLOv8-nano Baseline  |  Colab GPU (T4)

**Before running:**
1. Set Runtime → T4 GPU (`Runtime > Change runtime type > T4 GPU`)
2. Upload `data/processed/dataset_split_70_15_15.zip` to Google Drive at `MyDrive/CV/dataset_split_70_15_15.zip`
   - Generate it locally with: `python scripts/zip_split_for_upload.py`

**Outputs saved to Drive:** `MyDrive/CV/yolo_e1_results/`
**Download locally:** run the final cell — downloads `yolo_e1_results.zip`
Then extract locally: `python scripts/extract_colab_results.py yolo_e1_results.zip`

In [ ]:
# ── STEP 1: Mount Drive + install ────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "ultralytics>=8.0.0", "-q"])
print("ultralytics ready")

import os, shutil, time, zipfile, yaml
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── STEP 2: Extract dataset split from Drive ─────────────────────────────────
DRIVE        = Path("/content/drive/MyDrive/CV")
SPLIT_ZIP    = DRIVE / "dataset_split_70_15_15.zip"
SPLIT_OUT    = Path("/content/dataset_split_70_15_15")
COLAB_TRAIN  = Path("/content/yolo_e1")

assert SPLIT_ZIP.exists(), (
    f"Zip not found at {SPLIT_ZIP}\n"
    "Run locally: python scripts/zip_split_for_upload.py\n"
    "Then upload to Google Drive at MyDrive/CV/dataset_split_70_15_15.zip"
)

# Check for actual image directory — not just data.yaml — to detect partial extractions
if not (SPLIT_OUT / "train" / "images").exists():
    print(f"Extracting {SPLIT_ZIP.name}  ({SPLIT_ZIP.stat().st_size/1e6:.0f} MB) ...")
    with zipfile.ZipFile(SPLIT_ZIP, "r") as zf:
        zf.extractall("/content")
    print("Extracted.")
else:
    print("Split already extracted.")

# Rebuild data.yaml from scratch with Colab-local absolute paths
# (the local copy contains a Windows path that breaks YOLO on Colab)
DATA_YAML = SPLIT_OUT / "data.yaml"
cfg = {
    "path" : str(SPLIT_OUT),
    "train": "train/images",
    "val"  : "val/images",
    "test" : "test/images",
    "nc"   : 6,
    "names": ["BIODEGRADABLE", "CARDBOARD", "GLASS", "METAL", "PAPER", "PLASTIC"],
}
with open(DATA_YAML, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

# Verify image directories exist before YOLO tries to find them
for sp in ["train", "val", "test"]:
    img_dir = SPLIT_OUT / sp / "images"
    assert img_dir.exists(), f"Missing image directory: {img_dir}"

# Verify counts
print(f"\n{'Split':<8} {'Images':>8} {'Labels':>8}")
print("-" * 28)
for sp in ["train", "val", "test"]:
    ni = len(list((SPLIT_OUT / sp / "images").glob("*.jpg")))
    nl = len(list((SPLIT_OUT / sp / "labels").glob("*.txt")))
    flag = "" if ni == nl else "  <- MISMATCH"
    print(f"{sp:<8} {ni:>8} {nl:>8}{flag}")
print(f"\ndata.yaml: {DATA_YAML}")
print("data.yaml contents:")
print(open(DATA_YAML).read())

In [ ]:
# ── STEP 3: Train YOLOv8-nano on T4 ─────────────────────────────────────────
from ultralytics import YOLO

SEED = 42
model = YOLO("yolov8n.pt")
model.train(
    data       = str(DATA_YAML),
    epochs     = 120,           # more room to converge
    imgsz      = 640,           # standard YOLO resolution (was 416 — too small)
    batch      = 16,
    patience   = 25,
    project    = str(COLAB_TRAIN),
    name       = "E1_baseline_v2",
    seed       = SEED,
    exist_ok   = True,
    freeze     = 10,            # keep backbone frozen — dataset too small for full fine-tune
    device     = 0,
    verbose    = True,
    # Learning rate — low LR is critical when backbone is frozen
    lr0        = 0.001,         # was 0.01 — too high for frozen-backbone fine-tune
    lrf        = 0.01,          # cosine decay to lr0*lrf = 0.00001
    # Augmentation
    hsv_h=0.015, hsv_s=0.7,  hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5,
    perspective=0.001,
    flipud=0.0,  fliplr=0.5,
    mosaic=1.0,  mixup=0.05,
    copy_paste=0.0,             # was 0.1 — requires instance masks, bbox-only dataset
    close_mosaic=15,            # disable mosaic for final 15 epochs to stabilise
    conf=0.25,   iou=0.45,
)
TRAIN_RUN = COLAB_TRAIN / "E1_baseline_v2"
print(f"\nTraining complete: {TRAIN_RUN}")


In [ ]:
# ── STEP 4: Evaluate on test set ─────────────────────────────────────────────
TRAIN_RUN  = COLAB_TRAIN / "E1_baseline_v2"
BEST_PT    = TRAIN_RUN / "weights" / "best.pt"

best_model   = YOLO(str(BEST_PT))
test_metrics = best_model.val(
    data=str(DATA_YAML), split="test",
    imgsz=640, conf=0.25, iou=0.45, verbose=True,
)
rd = test_metrics.results_dict
print("\nTest metrics:")
for k, v in rd.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


In [ ]:
# ── STEP 5: FPS benchmark ────────────────────────────────────────────────────
test_images = sorted((SPLIT_OUT / "test" / "images").glob("*.jpg"))
for _ in range(2):
    best_model.predict(str(test_images[0]), conf=0.25, iou=0.45, verbose=False)

BENCH_N  = min(100, len(test_images))
times_ms = []
for img_path in test_images[:BENCH_N]:
    t0 = time.perf_counter()
    best_model.predict(str(img_path), conf=0.25, iou=0.45, verbose=False)
    times_ms.append((time.perf_counter() - t0) * 1000)

inference_ms = float(np.mean(times_ms))
fps          = 1000.0 / inference_ms
model_mb     = BEST_PT.stat().st_size / (1024 * 1024)
print(f"Inference (mean {BENCH_N} imgs): {inference_ms:.1f} ms")
print(f"FPS   : {fps:.1f}")
print(f"Model : {model_mb:.2f} MB")

In [ ]:
# ── STEP 6: Build all result files ───────────────────────────────────────────
CLASSES = ["BIODEGRADABLE", "CARDBOARD", "GLASS", "METAL", "PAPER", "PLASTIC"]

# Per-class mAP
class_indices    = list(test_metrics.box.ap_class_index)
per_class_ap50   = list(test_metrics.box.ap50)
per_class_ap5095 = list(test_metrics.box.maps)
ap50_by_cls      = {c: float("nan") for c in CLASSES}
ap5095_by_cls    = {c: float("nan") for c in CLASSES}
for idx, a50, a5095 in zip(class_indices, per_class_ap50, per_class_ap5095):
    ap50_by_cls[CLASSES[idx]]   = round(float(a50),   4)
    ap5095_by_cls[CLASSES[idx]] = round(float(a5095), 4)

training_log = pd.read_csv(TRAIN_RUN / "results.csv")

detection_row = {
    "experiment"    : "E1",
    "model"         : "yolov8n",
    "timestamp"     : datetime.now().strftime("%Y-%m-%d %H:%M"),
    "mAP50"         : round(float(rd.get("metrics/mAP50(B)",    float("nan"))), 4),
    "mAP50_95"      : round(float(rd.get("metrics/mAP50-95(B)", float("nan"))), 4),
    "precision"     : round(float(rd.get("metrics/precision(B)",float("nan"))), 4),
    "recall"        : round(float(rd.get("metrics/recall(B)",   float("nan"))), 4),
    "fps"           : round(fps, 2),
    "inference_ms"  : round(inference_ms, 2),
    "model_size_mb" : round(model_mb, 3),
    "epochs_trained": len(training_log),
    "image_size"    : 640,
    "dataset_split" : "70/15/15",
    **{f"mAP50_{c}":   ap50_by_cls[c]   for c in CLASSES},
    **{f"mAP5095_{c}": ap5095_by_cls[c] for c in CLASSES},
}

# Predictions CSV — stream to avoid OOM
pred_records = []
for result in best_model.predict([str(p) for p in test_images],
                                  conf=0.25, iou=0.45, verbose=False, stream=True):
    img_name = Path(result.path).name
    for box in result.boxes:
        cid = int(box.cls.item())
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        pred_records.append({"image": img_name, "class_id": cid,
                              "class_name": CLASSES[cid],
                              "confidence": round(float(box.conf.item()), 4),
                              "x1": round(x1,1), "y1": round(y1,1),
                              "x2": round(x2,1), "y2": round(y2,1)})

print(f"Predictions: {len(pred_records)}")


In [ ]:
# ── STEP 7: Save everything to Drive + package for local download ─────────────
# Folder structure in Drive mirrors the local repo layout exactly
RESULTS_ROOT = DRIVE / "yolo_e1_results"

SAVE_DIRS = {
    "models/yolo"                          : RESULTS_ROOT / "models" / "yolo",
    "models/checkpoints/last"             : RESULTS_ROOT / "models" / "checkpoints" / "last",
    "results/metrics"                     : RESULTS_ROOT / "results" / "metrics",
    "results/predictions"                 : RESULTS_ROOT / "results" / "predictions",
    "results/logs/training_logs"          : RESULTS_ROOT / "results" / "logs" / "training_logs",
    "figures/yolo"                        : RESULTS_ROOT / "figures" / "yolo",
}
for d in SAVE_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

# ── Weights ───────────────────────────────────────────────────────────────────
LAST_PT = TRAIN_RUN / "weights" / "last.pt"
shutil.copy2(BEST_PT, SAVE_DIRS["models/yolo"] / "yolo_baseline_E1.pt")
shutil.copy2(LAST_PT, SAVE_DIRS["models/checkpoints/last"] / "yolo_E1_last.pt")
print("Saved: yolo_baseline_E1.pt  +  yolo_E1_last.pt")

# ── Training log ──────────────────────────────────────────────────────────────
shutil.copy2(TRAIN_RUN / "results.csv",
             SAVE_DIRS["results/logs/training_logs"] / "E1_training_results.csv")
print("Saved: E1_training_results.csv")

# ── Metrics CSVs ──────────────────────────────────────────────────────────────
metrics_dir = SAVE_DIRS["results/metrics"]

# detection_results.csv (append/replace E1 row)
csv_path = metrics_dir / "detection_results.csv"
if csv_path.exists():
    existing = pd.read_csv(csv_path)
    existing = existing[existing["experiment"] != "E1"]
    df_out   = pd.concat([existing, pd.DataFrame([detection_row])], ignore_index=True)
else:
    df_out = pd.DataFrame([detection_row])
df_out.to_csv(csv_path, index=False)

pd.DataFrame([{"experiment": "E1", "class": c,
               "mAP50": ap50_by_cls[c], "mAP50_95": ap5095_by_cls[c]}
              for c in CLASSES]).to_csv(metrics_dir / "E1_per_class_metrics.csv", index=False)

pd.DataFrame(pred_records).to_csv(
    SAVE_DIRS["results/predictions"] / "E1_yolo_predictions.csv", index=False)
print("Saved: detection_results.csv  E1_per_class_metrics.csv  E1_yolo_predictions.csv")

# ── Figures ───────────────────────────────────────────────────────────────────
copied = 0
for pattern in ["*.png", "val_batch*.jpg"]:
    for src in TRAIN_RUN.glob(pattern):
        shutil.copy2(src, SAVE_DIRS["figures/yolo"] / src.name)
        copied += 1
for src in TRAIN_RUN.rglob("confusion_matrix*.png"):
    dst = SAVE_DIRS["figures/yolo"] / src.name
    if not dst.exists():
        shutil.copy2(src, dst); copied += 1
print(f"Saved: {copied} training figures")

print(f"\nAll results in Drive: {RESULTS_ROOT}")

In [ ]:
# ── STEP 8: Zip results for local download ────────────────────────────────────
# Creates yolo_e1_results.zip in /content — download it and run:
#   python scripts/extract_colab_results.py yolo_e1_results.zip

ZIP_LOCAL = Path("/content/yolo_e1_results.zip")

with zipfile.ZipFile(ZIP_LOCAL, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(RESULTS_ROOT.rglob("*")):
        if f.is_file():
            # arcname is relative to RESULTS_ROOT so extraction is easy
            zf.write(f, f.relative_to(RESULTS_ROOT))

size_mb = ZIP_LOCAL.stat().st_size / (1024 * 1024)
print(f"Created: {ZIP_LOCAL}  ({size_mb:.1f} MB)")

from google.colab import files
files.download(str(ZIP_LOCAL))
print("Download started — check your browser downloads.")

In [ ]:
# ── FINAL SUMMARY ────────────────────────────────────────────────────────────
print("=" * 60)
print("E1  YOLOv8-nano  |  T4 GPU  |  RESULTS")
print("=" * 60)
print(f"  mAP@0.5        : {detection_row['mAP50']:.4f}")
print(f"  mAP@0.5:0.95   : {detection_row['mAP50_95']:.4f}")
print(f"  Precision      : {detection_row['precision']:.4f}")
print(f"  Recall         : {detection_row['recall']:.4f}")
print(f"  FPS (T4)       : {detection_row['fps']:.1f}")
print(f"  Inference/img  : {detection_row['inference_ms']:.1f} ms")
print(f"  Model size     : {detection_row['model_size_mb']:.2f} MB")
print(f"  Epochs trained : {detection_row['epochs_trained']}")
print()
print("  Per-class mAP@0.5:")
for c in CLASSES:
    v   = ap50_by_cls[c]
    bar = "#" * int(v * 40) if v == v else "(nan)"
    print(f"    {c:<16} {v:.4f}  {bar}")
print()
print("  To extract locally:")
print("    python scripts/extract_colab_results.py yolo_e1_results.zip")